In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    average_precision_score, roc_auc_score, 
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from joblib import parallel_backend
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning)

# LOAD DATA

print("Loading data...")
df_train = pd.read_csv("/Users/stephenwillis/Desktop/archive/cleaned_loan_data_dev.csv")
df_eval = pd.read_csv("/Users/stephenwillis/Desktop/archive/cleaned_loan_data_holdout.csv")

# Set index for both dataframes
df_train = df_train.set_index("Id")
df_eval = df_eval.set_index("Id")

# Split features and target
X_train = df_train.drop(columns=["Defaulted"])
y_train = df_train["Defaulted"]

X_eval = df_eval.drop(columns=["Defaulted"])
y_eval = df_eval["Defaulted"]  

print(f"Training set shape: {X_train.shape}")
print(f"Holdout set shape: {X_eval.shape}")
print(f"Class distribution in training: {y_train.value_counts().to_dict()}")
print(f"Class distribution in holdout: {y_eval.value_counts().to_dict()}")


# PREPROCESSING PIPELINE


class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, features, eps=1e-4):
        self.features = features
        self.eps = eps
        self.woe_maps_ = {}
    
    def fit(self, X, y):
        for col in self.features:
            df = pd.concat([X[col], y], axis=1)
            grouped = df.groupby(col)[y.name].agg(['sum', 'count'])
            grouped["nevent"] = grouped["count"] - grouped["sum"]
            event_dist = grouped["sum"] / grouped["sum"].sum()
            nevent_dist = grouped["nevent"] / grouped["nevent"].sum()
            self.woe_maps_[col] = np.log((event_dist + self.eps) /
                                         (nevent_dist + self.eps)).to_dict()
        return self
    
    def transform(self, X):
        X_enc = X.copy()
        for col, mapping in self.woe_maps_.items():
            X_enc[col] = X_enc[col].map(mapping)
        return X_enc
    
    def set_output(self, *, transform=None):
        return self

# Define column types
num_cols = ["Term", "NoEmp", "ApprovalFY", "UrbanRural", "RevLineCr", 
            "LowDoc", "GrAppv", "SBA_Appv", "ExistingBusiness"]
cat_cols = [col for col in X_train.columns if col not in num_cols]

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='median')),
    ("scale", RobustScaler())
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy='most_frequent')),
    ("encode", WOEEncoder(features=cat_cols)),
    ("fill", SimpleImputer(strategy="constant", fill_value=0))
])

preproc = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
], remainder="drop")

preproc.set_output(transform="pandas")


# MODEL CONFIGURATION 

n_total = os.cpu_count()
n_best = max(1, n_total-1)

# Best Random Forest parameters
best_rf_params = {
    "n_jobs": n_best,
    "max_features": "log2",
    "max_samples": 0.5,
    "n_estimators": 100,
    "random_state": 42
}

# Best BorderlineSMOTE parameters
best_bsmote_params = {
    "sampling_strategy": 0.5,  
    "m_neighbors": 3,           
    "random_state": 42
}

# BUILD AND TRAIN FINAL MODEL

print("\nBuilding pipeline with BorderlineSMOTE...")

# Create the final pipeline with best configuration
final_pipeline = ImbPipeline([
    ("preproc", preproc),
    ("BorderlineSMOTE", BorderlineSMOTE(**best_bsmote_params)),
    ("model", RandomForestClassifier(**best_rf_params))
])

# Train on full dev set
print("Training model...")
with parallel_backend("loky"):
    final_pipeline.fit(X_train, y_train)

print("Training complete!")


# EVALUATE ON HOLDOUT SET

print("\nEvaluating on holdout set...")

# Get predictions and probabilities
y_pred = final_pipeline.predict(X_eval)
y_proba = final_pipeline.predict_proba(X_eval)[:, 1]

# Calculate metrics
metrics = {
    "Average Precision": average_precision_score(y_eval, y_proba),
    "ROC AUC": roc_auc_score(y_eval, y_proba),
    "Precision": precision_score(y_eval, y_pred),
    "Recall": recall_score(y_eval, y_pred),
    "F1 Score": f1_score(y_eval, y_pred)
}

# Print results
print("\n" + "="*50)
print("HOLDOUT SET PERFORMANCE")
print("="*50)
for metric_name, value in metrics.items():
    print(f"{metric_name:.<25} {value:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_eval, y_pred)
print(f"\nConfusion Matrix:")
print(f"TN: {cm[0,0]:,}  FP: {cm[0,1]:,}")
print(f"FN: {cm[1,0]:,}  TP: {cm[1,1]:,}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_eval, y_pred, 
                          target_names=['No Default', 'Default'],
                          digits=4))

# SAVE MODEL AND RESULTS

import joblib

# Save the trained model
joblib.dump(final_pipeline, 'final_bsmote_rf_model.pkl')
print("\nModel saved to: final_bsmote_rf_model.pkl")

# Save predictions
predictions_df = pd.DataFrame({
    'true_label': y_eval,
    'predicted_label': y_pred,
    'default_probability': y_proba
})
predictions_df.to_csv('holdout_predictions.csv', index=True)
print("Predictions saved to: holdout_predictions.csv")

print("\n" + "="*50)
print("ANALYSIS COMPLETE!")
print("="*50)

Loading data...
Training set shape: (807450, 17)
Holdout set shape: (89717, 17)
Class distribution in training: {0.0: 665648, 1.0: 141802}
Class distribution in holdout: {0.0: 73961, 1.0: 15756}

Building pipeline with BorderlineSMOTE...
Training model...
Training complete!

Evaluating on holdout set...

HOLDOUT SET PERFORMANCE
Average Precision........ 0.9115
ROC AUC.................. 0.9744
Precision................ 0.8611
Recall................... 0.8457
F1 Score................. 0.8533

Confusion Matrix:
TN: 71,812  FP: 2,149
FN: 2,431  TP: 13,325

Classification Report:
              precision    recall  f1-score   support

  No Default     0.9673    0.9709    0.9691     73961
     Default     0.8611    0.8457    0.8533     15756

    accuracy                         0.9490     89717
   macro avg     0.9142    0.9083    0.9112     89717
weighted avg     0.9486    0.9490    0.9488     89717


Model saved to: final_bsmote_rf_model.pkl
Predictions saved to: holdout_predictions.csv

A